# 02 — Data Cleaning

**Project:** TTC Transit Reliability Analytics
**Stage:** 2 — Cleaning
**Goal of this notebook:** turn the 12 raw event files (3 modes × 4 years) plus 3 code-description lookups into a single standardised fact table that the SQL stage and Power BI can both consume without any awareness of the raw formats.

This notebook implements the 7-step plan written at the end of `notebooks/01_data_audit.ipynb` (§9.9). Every step quoted there appears as a section below.

### Reading order

Each section opens with a plain-language explanation of *what* it does and *why*. The audit found two awkward realities the cleaning step must resolve, and they show up explicitly here:

1. **Bus & streetcar swap `Incident` text for an alphanumeric `Code` between 2024 and 2025.** We unify both into a single `delay_description` field — by reading the text directly for 2022–2024 and by joining `Code Descriptions.csv` for 2025+ (subway joins for every year). See [`docs/decisions.md`](../docs/decisions.md), Decision 9.
2. **Bus 2025+ stores `Line` as `"<route number> <route name>"`** (e.g. `"102 MARKHAM ROAD"`) while bus 2022–2024 stores the route number alone (e.g. `"320"`). Same for streetcar. We standardise on the route number so groupings are consistent across years. The route name is decorative; the number is the join key.

### Output

After running, `data/processed/` will contain:

| File | Rows | Purpose |
|---|---|---|
| `fact_subway.csv` | ~97k | Per-mode cleaned fact (debugging) |
| `fact_bus.csv` | ~244k | Per-mode cleaned fact (debugging) |
| `fact_streetcar.csv` | ~62k | Per-mode cleaned fact (debugging) |
| `fact_delay_events.csv` | ~403k | Combined fact — this is what Postgres and Power BI load |

Per the choices made at the start of this stage: per-mode files exist for diagnostic ease, and a single combined file exists as the canonical downstream input.

### Standardised schema

Every cleaned row, regardless of source, has these columns (and only these columns):

```
event_id            int   (sequence assigned after concat, 1-based)
transit_mode        str   ("Subway", "Bus", "Streetcar")
source_file         str   (raw filename for traceability)

event_date          date
event_time          str   ("HH:MM"; kept as string because some sources have no seconds)
event_datetime      timestamp  (event_date + event_time)
year, month         int
month_name          str
day_of_week         str   (derived from event_datetime, not the raw Day column)
hour                int

line                str   (subway: YU/BD/SHP/SRT; bus: route number; streetcar: route number)
station             str   (subway: station name; bus/streetcar: location-ish description)
bound               str   (N/E/S/W, may be null — see audit §9.7)

delay_code          str   (alphanumeric TTC code; null for bus/streetcar 2022-2024)
delay_description   str   (always populated; from Incident text or code lookup)

delay_minutes       int
gap_minutes         int
vehicle             int
```

Derived columns deliberately *not* added here (left for the SQL/dimension stage):
`is_major_delay` (no defended threshold yet), `delay_category` (no defended grouping rule yet). See the Stage 2 choice made before this notebook was started.


## 0. Setup

In [1]:
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw dir:     ", RAW)
print("Processed:   ", PROCESSED)


Project root: C:\Users\soham\Desktop\Coding Stuff\TTC Transit Delay & Reliability Analysis
Raw dir:      C:\Users\soham\Desktop\Coding Stuff\TTC Transit Delay & Reliability Analysis\data\raw
Processed:    C:\Users\soham\Desktop\Coding Stuff\TTC Transit Delay & Reliability Analysis\data\processed


## 1. Target schema (canonical column order)

Defining the column order once means every per-mode cleaner can reindex against `TARGET_COLUMNS` at the end, and any missing column shows up as a hard column-not-found error instead of silently producing a misaligned CSV.


In [2]:
TARGET_COLUMNS = [
    "event_id",
    "transit_mode",
    "source_file",
    "event_date",
    "event_time",
    "event_datetime",
    "year",
    "month",
    "month_name",
    "day_of_week",
    "hour",
    "line",
    "station",
    "bound",
    "delay_code",
    "delay_description",
    "delay_minutes",
    "gap_minutes",
    "vehicle",
]
print(f"{len(TARGET_COLUMNS)} columns in the standardised fact table")


19 columns in the standardised fact table


## 2. Helper functions

Five small functions, defined once. Naming them clearly is the documentation:

- `read_excel_stacked(path)` — reuse from the audit. Reads every sheet and concatenates. The audit proved every XLSX is single-sheet today, but using `sheet_name=None` future-proofs against TTC ever shipping monthly tabs again (see Decision 8).
- `extract_route_number(s)` — pulls the leading integer from strings like `"102 MARKHAM ROAD"` → `"102"`. Used to normalise bus & streetcar 2025+ `Line` values down to just the route number, matching the 2022–2024 convention.
- `attach_codes(df, lookup, mode)` — left-joins a code-description lookup onto rows whose `delay_description` is still null (i.e. rows where the source only gave us a code, not text).
- `add_time_decomposition(df)` — builds `event_datetime` from date + time and derives `year`, `month`, `month_name`, `day_of_week`, `hour`.
- `finalise(df)` — applies the canonical column order, handles the `XXXXX` placeholder, and fills any remaining unmatched codes with `Unknown ({code})` so no row has a null `delay_description`.


In [3]:
def read_excel_stacked(path: Path) -> pd.DataFrame:
    sheets = pd.read_excel(path, sheet_name=None)
    parts = []
    for name, part in sheets.items():
        part = part.copy()
        part["_sheet"] = name
        parts.append(part)
    return pd.concat(parts, ignore_index=True, sort=False)


_NUM_PREFIX = re.compile(r"^(\d+)")

def extract_route_number(value) -> str:
    """'102 MARKHAM ROAD' -> '102'. Non-numeric prefixes are kept as-is."""
    if pd.isna(value):
        return value
    s = str(value).strip()
    m = _NUM_PREFIX.match(s)
    return m.group(1) if m else s


def attach_codes(df: pd.DataFrame, lookup: pd.DataFrame, mode: str) -> pd.DataFrame:
    """Populate delay_description from the code lookup for any row that has a
    delay_code but no description yet. Bus/streetcar 2022-2024 already have
    delay_description (from Incident text) and a null delay_code, so they pass
    through untouched.
    """
    lk = lookup.rename(columns={"CODE": "delay_code", "DESCRIPTION": "_desc_from_lookup"})
    lk = lk[["delay_code", "_desc_from_lookup"]].drop_duplicates(subset="delay_code")
    out = df.merge(lk, on="delay_code", how="left")
    needs_desc = out["delay_description"].isna() & out["_desc_from_lookup"].notna()
    out.loc[needs_desc, "delay_description"] = out.loc[needs_desc, "_desc_from_lookup"]
    return out.drop(columns=["_desc_from_lookup"])


def add_time_decomposition(df: pd.DataFrame) -> pd.DataFrame:
    """Build event_datetime from event_date + event_time and derive
    year / month / month_name / day_of_week / hour from it. We compute
    day_of_week ourselves rather than trusting the raw 'Day' column."""
    date = pd.to_datetime(df["event_date"], errors="coerce")
    time_str = df["event_time"].astype(str).fillna("00:00")
    # event_time can arrive as '02:30' or '02:30:00'; coerce both.
    dt = pd.to_datetime(date.dt.strftime("%Y-%m-%d") + " " + time_str, errors="coerce")
    return df.assign(
        event_date=date.dt.date,
        event_datetime=dt,
        year=dt.dt.year,
        month=dt.dt.month,
        month_name=dt.dt.month_name(),
        day_of_week=dt.dt.day_name(),
        hour=dt.dt.hour,
    )


def finalise(df: pd.DataFrame, mode: str) -> pd.DataFrame:
    """Apply the canonical column order, scrub the XXXXX placeholder, and
    fill any still-unmatched code with 'Unknown ({code})' for honest
    downstream display."""
    # XXXXX is a known placeholder in the subway source for 'no real code'.
    placeholder = df["delay_code"] == "XXXXX"
    n_placeholder = int(placeholder.sum())
    if n_placeholder:
        df.loc[placeholder, "delay_description"] = "Unknown (XXXXX placeholder)"
        df.loc[placeholder, "delay_code"] = pd.NA

    # Any remaining row that has a code but still no description: label it.
    needs_label = df["delay_code"].notna() & df["delay_description"].isna()
    df.loc[needs_label, "delay_description"] = "Unknown (" + df.loc[needs_label, "delay_code"].astype(str) + ")"

    # Drop helper columns and reindex to canonical order. Use missing-safe reindex.
    for col in TARGET_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA
    return df[TARGET_COLUMNS].copy()


## 3. Subway

Subway is the easiest of the three modes because its schema is stable from 2022 through 2025+. The only renames are `Date→event_date`, `Time→event_time`, `Station→station`, `Code→delay_code`, `Min Delay→delay_minutes`, `Min Gap→gap_minutes`, `Bound→bound`, `Line→line`, `Vehicle→vehicle`. The `_id` column in the 2025+ CSV is dropped (it's just a row index).


In [4]:
SUBWAY_RENAME = {
    "Date": "event_date",
    "Time": "event_time",
    "Station": "station",
    "Code": "delay_code",
    "Min Delay": "delay_minutes",
    "Min Gap": "gap_minutes",
    "Bound": "bound",
    "Line": "line",
    "Vehicle": "vehicle",
}

def clean_subway() -> pd.DataFrame:
    lookup = pd.read_csv(RAW / "subway" / "Code Descriptions.csv")
    paths = [
        RAW / "subway" / "ttc-subway-delay-data-2022.xlsx",
        RAW / "subway" / "ttc-subway-delay-data-2023.xlsx",
        RAW / "subway" / "ttc-subway-delay-data-2024.xlsx",
        RAW / "subway" / "TTC Subway Delay Data since 2025.csv",
    ]
    parts = []
    for p in paths:
        df = read_excel_stacked(p) if p.suffix == ".xlsx" else pd.read_csv(p)
        df = df.rename(columns=SUBWAY_RENAME)
        df["transit_mode"] = "Subway"
        df["source_file"] = p.name
        df["delay_description"] = pd.NA  # will be filled by attach_codes
        parts.append(df)
    df = pd.concat(parts, ignore_index=True, sort=False)
    df = attach_codes(df, lookup, "Subway")
    df = add_time_decomposition(df)
    df = finalise(df, "Subway")
    return df


subway = clean_subway()
print(f"Subway cleaned: {len(subway):,} rows")
print(subway.head(3))


Subway cleaned: 97,502 rows
  event_id transit_mode                      source_file  event_date event_time      event_datetime  year  month month_name day_of_week  hour line                 station bound delay_code  \
0     <NA>       Subway  ttc-subway-delay-data-2022.xlsx  2022-01-01      15:59 2022-01-01 15:59:00  2022      1    January    Saturday    15  SRT   LAWRENCE EAST STATION     N       SRDP   
1     <NA>       Subway  ttc-subway-delay-data-2022.xlsx  2022-01-01      02:23 2022-01-01 02:23:00  2022      1    January    Saturday     2   BD      SPADINA BD STATION   NaN       MUIS   
2     <NA>       Subway  ttc-subway-delay-data-2022.xlsx  2022-01-01      22:00 2022-01-01 22:00:00  2022      1    January    Saturday    22  SRT  KENNEDY SRT STATION TO   NaN        MRO   

                               delay_description  delay_minutes  gap_minutes  vehicle  
0                                 Unknown (SRDP)              0            0     3023  
1  INJURED/ILL CUSTOMER IN STAT

## 4. Bus

Bus is the most invasive case. The 2022–2024 XLSX files store delay reasons as **free-text** `Incident` strings and never carry a code; the 2025+ CSV stores an alphanumeric `Code` and the description must be joined from the lookup. Column names also differ — `Route` becomes `Line`, `Location` becomes `Station`, `Direction` becomes `Bound`. The 2025+ `Line` column has the form `"102 MARKHAM ROAD"`; we strip it down to just the route number so it groups consistently with the 2022–2024 `Route` values.


In [5]:
BUS_RENAME_XLSX = {
    "Date": "event_date",
    "Route": "line",
    "Time": "event_time",
    "Location": "station",
    "Incident": "delay_description",
    "Min Delay": "delay_minutes",
    "Min Gap": "gap_minutes",
    "Direction": "bound",
    "Vehicle": "vehicle",
}

BUS_RENAME_CSV = {
    "Date": "event_date",
    "Line": "line",
    "Time": "event_time",
    "Station": "station",
    "Code": "delay_code",
    "Min Delay": "delay_minutes",
    "Min Gap": "gap_minutes",
    "Bound": "bound",
    "Vehicle": "vehicle",
}

def clean_bus() -> pd.DataFrame:
    lookup = pd.read_csv(RAW / "bus" / "Code Descriptions.csv")
    xlsx_paths = [
        RAW / "bus" / "ttc-bus-delay-data-2022.xlsx",
        RAW / "bus" / "ttc-bus-delay-data-2023.xlsx",
        RAW / "bus" / "ttc-bus-delay-data-2024.xlsx",
    ]
    csv_path = RAW / "bus" / "TTC Bus Delay Data since 2025.csv"

    parts = []

    # 2022-2024: Incident text directly becomes delay_description; delay_code stays NA.
    for p in xlsx_paths:
        df = read_excel_stacked(p).rename(columns=BUS_RENAME_XLSX)
        df["delay_code"] = pd.NA
        df["line"] = df["line"].astype(str)
        df["transit_mode"] = "Bus"
        df["source_file"] = p.name
        parts.append(df)

    # 2025+: alphanumeric Code joins to lookup; Line is "<num> <name>", strip to number.
    csv = pd.read_csv(csv_path).rename(columns=BUS_RENAME_CSV)
    csv["line"] = csv["line"].apply(extract_route_number)
    csv["delay_description"] = pd.NA
    csv["transit_mode"] = "Bus"
    csv["source_file"] = csv_path.name
    parts.append(csv)

    df = pd.concat(parts, ignore_index=True, sort=False)
    df = attach_codes(df, lookup, "Bus")
    df = add_time_decomposition(df)
    df = finalise(df, "Bus")
    return df


bus = clean_bus()
print(f"Bus cleaned: {len(bus):,} rows")
print(bus.head(3))


Bus cleaned: 243,594 rows
  event_id transit_mode                   source_file  event_date event_time      event_datetime  year  month month_name day_of_week  hour line                 station bound delay_code  \
0     <NA>          Bus  ttc-bus-delay-data-2022.xlsx  2022-01-01      02:00 2022-01-01 02:00:00  2022      1    January    Saturday     2  320        YONGE AND DUNDAS   NaN       <NA>   
1     <NA>          Bus  ttc-bus-delay-data-2022.xlsx  2022-01-01      02:00 2022-01-01 02:00:00  2022      1    January    Saturday     2  325  OVERLEA AND THORCLIFFE     W       <NA>   
2     <NA>          Bus  ttc-bus-delay-data-2022.xlsx  2022-01-01      02:00 2022-01-01 02:00:00  2022      1    January    Saturday     2  320       YONGE AND STEELES     S       <NA>   

       delay_description  delay_minutes  gap_minutes  vehicle  
0          General Delay              0            0     8531  
1              Diversion            131          161     8658  
2  Operations - Operator     

## 5. Streetcar

Streetcar is *semantically* closer to subway than bus was: it already calls its column `Line` (not `Route`) and its direction column `Bound` (not `Direction`). So the renames are slightly shorter than bus's — though we still have to lowercase `Line→line` and `Bound→bound` to match the target schema. Otherwise the same pattern as bus: 2022–2024 uses `Incident` text, 2025+ uses `Code` that must be joined, and the 2025+ `Line` carries a "<number> <name>" string that we trim down to the number.


In [6]:
STREETCAR_RENAME_XLSX = {
    "Date": "event_date",
    "Time": "event_time",
    "Line": "line",
    "Location": "station",
    "Bound": "bound",
    "Incident": "delay_description",
    "Min Delay": "delay_minutes",
    "Min Gap": "gap_minutes",
    "Vehicle": "vehicle",
}

STREETCAR_RENAME_CSV = {
    "Date": "event_date",
    "Time": "event_time",
    "Line": "line",
    "Station": "station",
    "Bound": "bound",
    "Code": "delay_code",
    "Min Delay": "delay_minutes",
    "Min Gap": "gap_minutes",
    "Vehicle": "vehicle",
}


def clean_streetcar() -> pd.DataFrame:
    lookup = pd.read_csv(RAW / "streetcar" / "Code Descriptions.csv")
    xlsx_paths = [
        RAW / "streetcar" / "ttc-streetcar-delay-data-2022.xlsx",
        RAW / "streetcar" / "ttc-streetcar-delay-data-2023.xlsx",
        RAW / "streetcar" / "ttc-streetcar-delay-data-2024.xlsx",
    ]
    csv_path = RAW / "streetcar" / "TTC Streetcar Delay Data since 2025.csv"

    parts = []

    for p in xlsx_paths:
        df = read_excel_stacked(p).rename(columns=STREETCAR_RENAME_XLSX)
        df["delay_code"] = pd.NA
        df["line"] = df["line"].astype(str)
        df["transit_mode"] = "Streetcar"
        df["source_file"] = p.name
        parts.append(df)

    csv = pd.read_csv(csv_path).rename(columns=STREETCAR_RENAME_CSV)
    csv["line"] = csv["line"].apply(extract_route_number)
    csv["delay_description"] = pd.NA
    csv["transit_mode"] = "Streetcar"
    csv["source_file"] = csv_path.name
    parts.append(csv)

    df = pd.concat(parts, ignore_index=True, sort=False)
    df = attach_codes(df, lookup, "Streetcar")
    df = add_time_decomposition(df)
    df = finalise(df, "Streetcar")
    return df


streetcar = clean_streetcar()
print(f"Streetcar cleaned: {len(streetcar):,} rows")
print(streetcar.head(3))


Streetcar cleaned: 62,059 rows
  event_id transit_mode                         source_file  event_date event_time      event_datetime  year  month month_name day_of_week  hour line            station bound delay_code  \
0     <NA>    Streetcar  ttc-streetcar-delay-data-2022.xlsx  2022-01-01      02:21 2022-01-01 02:21:00  2022      1    January    Saturday     2  504  BROADVIEW STATION     E       <NA>   
1     <NA>    Streetcar  ttc-streetcar-delay-data-2022.xlsx  2022-01-01      03:22 2022-01-01 03:22:00  2022      1    January    Saturday     3  501  718 QUEEN ST EAST     W       <NA>   
2     <NA>    Streetcar  ttc-streetcar-delay-data-2022.xlsx  2022-01-01      03:28 2022-01-01 03:28:00  2022      1    January    Saturday     3  504  BROADVIEW STATION     S       <NA>   

          delay_description  delay_minutes  gap_minutes  vehicle  
0  Collision - TTC Involved             30           60     8333  
1                Operations             16           35     8068  
2          

## 6. Combine and assign event_id

All three per-mode DataFrames share the canonical column order, so a plain `concat` is enough. After concatenating, we assign a project-wide unique `event_id` so every row in the combined fact table has a stable primary key for the SQL stage.


In [7]:
combined = pd.concat([subway, bus, streetcar], ignore_index=True)
combined["event_id"] = range(1, len(combined) + 1)
combined = combined[TARGET_COLUMNS]
print(f"Combined fact table: {len(combined):,} rows x {combined.shape[1]} columns")
print(combined.head(3))


Combined fact table: 403,155 rows x 19 columns
   event_id transit_mode                      source_file  event_date event_time      event_datetime  year  month month_name day_of_week  hour line                 station bound delay_code  \
0         1       Subway  ttc-subway-delay-data-2022.xlsx  2022-01-01      15:59 2022-01-01 15:59:00  2022      1    January    Saturday    15  SRT   LAWRENCE EAST STATION     N       SRDP   
1         2       Subway  ttc-subway-delay-data-2022.xlsx  2022-01-01      02:23 2022-01-01 02:23:00  2022      1    January    Saturday     2   BD      SPADINA BD STATION   NaN       MUIS   
2         3       Subway  ttc-subway-delay-data-2022.xlsx  2022-01-01      22:00 2022-01-01 22:00:00  2022      1    January    Saturday    22  SRT  KENNEDY SRT STATION TO   NaN        MRO   

                               delay_description  delay_minutes  gap_minutes  vehicle  
0                                 Unknown (SRDP)              0            0     3023  
1  INJUR

## 7. Validation checks

Five quick assertions a hostile reviewer would ask for:

1. **Row count matches the audit.** The audit found 403,155 total rows; the combined fact must also have 403,155.
2. **No nulls in critical columns.** `event_date`, `event_datetime`, `transit_mode`, `delay_description` must be 100% populated.
3. **Every transit_mode value is one of {Subway, Bus, Streetcar}.**
4. **Delay minutes are non-negative.** Negative values would indicate a sign-flip or bad upstream data.
5. **Bus/streetcar 2022–2024 rows have null `delay_code`; everything else has a non-null code** (except the XXXXX rows we deliberately nulled).


In [8]:
print("Row count check:")
print(f"  combined rows = {len(combined):,}  (audit said 403,155)")

print("\nNull counts in critical columns:")
for col in ["event_date", "event_datetime", "transit_mode", "delay_description"]:
    print(f"  {col:20s}  nulls = {combined[col].isna().sum():,}")

print("\ntransit_mode value counts:")
print(combined["transit_mode"].value_counts())

print("\nDelay minutes sanity:")
print(f"  min delay_minutes = {combined['delay_minutes'].min()}")
print(f"  max delay_minutes = {combined['delay_minutes'].max()}")
print(f"  min gap_minutes   = {combined['gap_minutes'].min()}")
print(f"  max gap_minutes   = {combined['gap_minutes'].max()}")

print("\ndelay_code nullness by mode + era:")
era = combined["source_file"].str.contains("since 2025").map({True: "2025+", False: "2022-2024"})
print(pd.crosstab([combined["transit_mode"], era], combined["delay_code"].isna(),
                  rownames=["mode", "era"], colnames=["delay_code_isna"]))


Row count check:
  combined rows = 403,155  (audit said 403,155)

Null counts in critical columns:
  event_date            nulls = 0
  event_datetime        nulls = 0
  transit_mode          nulls = 0
  delay_description     nulls = 0

transit_mode value counts:
transit_mode
Bus          243594
Subway        97502
Streetcar     62059
Name: count, dtype: int64

Delay minutes sanity:
  min delay_minutes = 0
  max delay_minutes = 999
  min gap_minutes   = 0
  max gap_minutes   = 999

delay_code nullness by mode + era:
delay_code_isna      False   True 
mode      era                     
Bus       2022-2024      0  174557
          2025+      69037       0
Streetcar 2022-2024      0   46274
          2025+      15775      10
Subway    2022-2024  69308       3
          2025+      28188       3


## 8. Write standardised CSVs to `data/processed/`

Per the Stage 2 choice: per-mode files (`fact_subway.csv`, `fact_bus.csv`, `fact_streetcar.csv`) for debugging plus one combined `fact_delay_events.csv` as the canonical downstream input.

`index=False` so we don't write pandas's row index as a column. `date_format="%Y-%m-%d"` keeps `event_date` ISO-formatted (Postgres-friendly).


In [9]:
for mode_name, df in [("subway", subway), ("bus", bus), ("streetcar", streetcar)]:
    out = PROCESSED / f"fact_{mode_name}.csv"
    df.to_csv(out, index=False, date_format="%Y-%m-%d %H:%M:%S")
    print(f"  wrote {out.relative_to(PROJECT_ROOT)}  ({len(df):,} rows, {out.stat().st_size / 1024:.0f} KB)")

combined_out = PROCESSED / "fact_delay_events.csv"
combined.to_csv(combined_out, index=False, date_format="%Y-%m-%d %H:%M:%S")
print(f"\n  wrote {combined_out.relative_to(PROJECT_ROOT)}  ({len(combined):,} rows, {combined_out.stat().st_size / 1024:.0f} KB)")


  wrote data\processed\fact_subway.csv  (97,502 rows, 16418 KB)


  wrote data\processed\fact_bus.csv  (243,594 rows, 36134 KB)
  wrote data\processed\fact_streetcar.csv  (62,059 rows, 9873 KB)



  wrote data\processed\fact_delay_events.csv  (403,155 rows, 64679 KB)


## 9. Summary

This notebook took 12 raw files in two different formats with three different per-mode schemas and reduced them to a single standardised fact table with the canonical columns listed in §1.

The four cleaned files in `data/processed/` are the only inputs the SQL stage will use. Nothing downstream needs to know that 2022–2024 was XLSX or that bus/streetcar had a `Code`-vs-`Incident` discontinuity.

### Next concrete step (Stage 3 — SQL modelling)

1. Write `sql/01_create_schema.sql` defining the `fact_delay_events` table with columns matching the CSV (and appropriate Postgres types).
2. Write a small Python loader using `pandas` + `SQLAlchemy` to bulk-insert the combined CSV into Postgres.
3. Write `sql/03_create_dimensions.sql` for `dim_date`, `dim_mode`, `dim_delay_cause` per the plan in `docs/project_plan.md` §5.
4. Write `sql/05_reporting_views.sql` for the analysis-ready views.

### What we deliberately left for later

- `is_major_delay` — still needs a defended threshold. Add as a derived column in a SQL view once we look at the distribution of `delay_minutes`.
- `delay_category` — still needs a defended grouping rule. Likely a join against a hand-built CSV mapping codes/descriptions to ~6 categories (mechanical / operations / incident / passenger / external / unknown).
